# Lesson 4 — Sensitivity, Duality, Shadow Prices

## Learning objectives

1. Derive the LP dual mechanically from the primal.
2. Interpret reduced costs and shadow prices.
3. Read a sensitivity report: ranging on RHS and on objective coefficients.
4. Use duality to *certify* optimality and detect degeneracy.

We assume Lesson 2's geometry of the polyhedron.

## 1. Mechanical derivation of the dual

For the canonical primal

$$\min c^\top x \;\;\text{s.t.}\;\; Ax \ge b, x \ge 0,$$

the dual is

$$\max b^\top y \;\;\text{s.t.}\;\; A^\top y \le c, y \ge 0.$$

The mapping (primal ↔ dual):

| Primal | Dual |
|--------|------|
| min | max |
| $\ge$ constraint | $y_i \ge 0$ |
| $=$ constraint | $y_i$ free |
| $x_j \ge 0$ | $\le$ constraint |
| $x_j$ free | $=$ constraint |

This is **mechanical**, but it codifies a deep fact: every degree of freedom in one problem is constrained by a constraint in the other {cite:p}`Bertsimas1997,Boyd2004`.

## 2. Shadow prices

The dual variable $y_i^\star$ is the **shadow price** of constraint $i$. It is the marginal change in optimal cost per unit relaxation of $b_i$:

$$\frac{\partial z^\star}{\partial b_i} = y_i^\star \quad \text{(within the basis range).}$$

**Two interpretations:**

- *Economic*: in production planning, $y_i$ tells you the maximum you'd pay for one more unit of resource $i$.
- *Algorithmic*: the simplex method uses dual values to decide which non-basic variable to bring in (entering variable rule).

## 3. Reduced costs

For non-basic variable $x_j$, the **reduced cost** $\bar c_j = c_j - A_j^\top y^\star$ is the *rate* at which the objective changes if you push $x_j$ above 0. Optimality means all reduced costs are non-negative (for a min problem); otherwise simplex pivots.

Reduced costs also give **objective ranging**: by how much can $c_j$ change before the current basis stops being optimal? Answer: until $\bar c_j$ would change sign.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

c = np.array([3.0, 2.0])
A = np.array([[1, 1], [2, 1], [1, 3]])
b = np.array([4.0, 5.0, 6.0])

m = do.Model("sens")
x = m.continuous("x", shape=(2,), lb=0)
# Build the three rows individually so each gets a stable name we can look
# up in r.constraint_duals.
m.subject_to(A[0, 0] * x[0] + A[0, 1] * x[1] >= b[0])
m.subject_to(A[1, 0] * x[0] + A[1, 1] * x[1] >= b[1])
m.subject_to(A[2, 0] * x[0] + A[2, 1] * x[1] >= b[2])
m.minimize(c[0] * x[0] + c[1] * x[1])
r = m.solve()
print("primal x =", r.value(x).round(4), "  obj =", round(r.objective, 4))

# discopt exposes KKT multipliers directly when the solver returns them:
#   r.constraint_duals      -- dict keyed by Constraint.name (shadow prices y_i)
#   r.bound_duals_lower     -- dict keyed by Variable.name (reduced costs at lb)
#   r.bound_duals_upper     -- dict keyed by Variable.name (reduced costs at ub)
# All values are in the internal-minimization sign convention.
print("shadow prices y* =", r.constraint_duals)
print("reduced costs at x>=0 =", r.bound_duals_lower)

# Cross-check b^T y == c^T x (zero duality gap):
y = np.array([float(r.constraint_duals[name]) for name in ("c0", "c1", "c2")])
print("b^T y =", round(float(b @ y), 4), "  c^T x =", round(float(c @ r.value(x)), 4))


## 4. Verifying optimality without resolving

You can certify a primal solution $\hat x$ is optimal by exhibiting a $\hat y$ with:

1. $A^\top \hat y \le c, \hat y \ge 0$ (dual feasibility),
2. $b^\top \hat y = c^\top \hat x$ (zero duality gap),
3. complementary slackness $\hat y_i \cdot (A_i \hat x - b_i) = 0$.

In B&B for MILP (Lesson 6), this is exactly how the LP relaxation provides a certified bound at each node.

## 5. Degeneracy

A BFS is **degenerate** if some basic variable is at its bound (typically 0). At a degenerate vertex, multiple bases describe the same point, so pivots may make zero progress (cycling). Bland's rule {cite:p}`Bland1977` and lexicographic perturbation prevent cycles. In modeling, degeneracy often signals **redundant constraints** — sometimes an opportunity for presolve {cite:p}`Savelsbergh1994`.

## References
{cite:p}`Bertsimas1997` Ch 4–5; {cite:p}`Boyd2004` Ch 5 (general convex duality); {cite:p}`Dantzig1963`.